# Archive Suite — Google Colab GPU Pack Runner (Google AI Ultra)

Runs any staged Archive Suite benchmark or training pack from `My Drive/ArchiveSuite-Colab/packs/<PACK_NAME>/`, writing resume-safe outputs directly to `My Drive/ArchiveSuite-Colab/results/<PACK_NAME>/`.

Supports:
- **`vllm-ocr`** packs (`candidates.json`, e.g. `ocr-family-coverage-rerun-20261126`) against frozen `public895` + `truth99` datasets
- **`yolo-marks`** packs (`pack-type.json`, e.g. `marks-detect-own-20261127`) fine-tuning YOLO11m from `base_weights/best.pt` and predicting on the 32 held-out eval pages
- Set `PACK_NAME = 'ALL'` to run every staged pack inside `packs/` back-to-back.

**Recommended runtime (Google AI Ultra):**
- **A100 80 GB / H100 80 GB + High-RAM** for 27B FP8 models or fast full sweeps.
- **L4 24 GB + High-RAM** for 1B–9B OCR models or YOLO11m fine-tuning.
- Enable **Background execution** so runs finish even if your laptop sleeps.

In [ ]:
# 1. Mount Google Drive, inspect GPU, and list packs
from google.colab import drive
drive.mount('/content/drive')

import json, os, subprocess
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ArchiveSuite-Colab')
PACK_NAME = 'ALL'        # Set a single folder name in packs/ or 'ALL'
VLLM_VERSION = '0.31.0'  # Pinned to match Sherlock's vllm/vllm-openai:v0.31.0 container

# Keep Hugging Face weights & caches on fast local Colab NVMe (never slow Drive)
os.environ['HF_HOME'] = '/content/hf'
os.environ['XDG_CACHE_HOME'] = '/content/cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
assert DRIVE_ROOT.is_dir(), f'Missing {DRIVE_ROOT} — upload ArchiveSuite-Colab to My Drive first.'
ALL_PACKS = sorted(p.name for p in (DRIVE_ROOT / 'packs').iterdir() if p.is_dir())
SELECTED_PACKS = ALL_PACKS if PACK_NAME == 'ALL' else [PACK_NAME]
print('Available packs:', ALL_PACKS)
print('Selected packs:', SELECTED_PACKS)

In [ ]:
# 2. Install only the dependencies required by SELECTED_PACKS
import json, sys, subprocess

need_vllm = False
need_mineru = False
need_yolo = False
for pname in SELECTED_PACKS:
    pdir = DRIVE_ROOT / 'packs' / pname
    meta = json.loads((pdir / 'colab-pack.json').read_text()) if (pdir / 'colab-pack.json').is_file() else {}
    ptype = meta.get('pack_type') or ('vllm-ocr' if (pdir / 'candidates.json').is_file() else 'generic')
    if ptype == 'vllm-ocr':
        need_vllm = True
        if meta.get('needs_mineru'):
            need_mineru = True
    elif ptype == 'yolo-marks':
        need_yolo = True

if need_vllm:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'vllm=={VLLM_VERSION}', 'huggingface_hub', 'pillow'])
    if need_mineru:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mineru-vl-utils'])
    import torch, vllm
    print('vLLM ready:', vllm.__version__, 'torch:', torch.__version__, 'cuda:', torch.cuda.is_available())

if need_yolo:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'pillow'])
    import ultralytics
    print('Ultralytics ready:', ultralytics.__version__)

In [ ]:
# 3. Run every selected pack (resume-safe: completed pages are never re-read)
import json, os, shutil, subprocess, sys
from pathlib import Path

def run_vllm_ocr_pack(pack_dir: Path, res_dir: Path):
    runners = Path('/content/runners') / pack_dir.name
    if runners.exists():
        shutil.rmtree(runners)
    shutil.copytree(pack_dir / 'colab_support', runners)

    cands_path = pack_dir / 'candidates.json'
    cands = json.loads(cands_path.read_text())
    root895 = res_dir / '895'
    root96 = res_dir / '96'
    root895.mkdir(parents=True, exist_ok=True)
    root96.mkdir(parents=True, exist_ok=True)

    # Copy Drive manifests into split roots so family_run finds pilot3.jsonl + manifest
    for split_root, ds_dir, manifest in [
        (root895, DRIVE_ROOT / 'datasets' / 'public895', 'public895.jsonl'),
        (root96, DRIVE_ROOT / 'datasets' / 'truth99', 'truth99.jsonl'),
    ]:
        for mf in ('pilot3.jsonl', manifest):
            dst = split_root / mf
            if not dst.exists():
                shutil.copy2(ds_dir / mf, dst)

    env = os.environ.copy()
    env['PYTHONPATH'] = str(runners) + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')

    # Download pinned snapshots onto local /content/hf and mark prepared.json state='ready'
    subprocess.check_call([sys.executable, str(runners / 'colab_prepare.py'), str(cands_path), str(root895), str(root96)], env=env)

    for entry in cands:
        key = entry['key']
        print(f'=== Running OCR candidate {key} ({entry.get("model")}) ===', flush=True)
        for split, split_root, manifest in [('895', root895, 'public895.jsonl'), ('96', root96, 'truth99.jsonl')]:
            if split not in entry.get('sets', ['895', '96']):
                continue
            sess = split_root / key
            run_env = dict(env, SESSION_ROOT=str(sess))
            subprocess.check_call([sys.executable, str(runners / 'family_run.py'), manifest], env=run_env)

def run_yolo_marks_pack(pack_dir: Path, res_dir: Path):
    cfg = json.loads((pack_dir / 'pack-type.json').read_text())
    # Copy pack to fast local /content/yolo_pack for fast image IO during training
    local_pack = Path('/content/yolo_packs') / pack_dir.name
    if local_pack.exists():
        shutil.rmtree(local_pack)
    shutil.copytree(pack_dir, local_pack)
    script = local_pack / cfg.get('script', 'marks_detect_own_cluster.py')
    local_out = Path('/content/yolo_out') / pack_dir.name
    local_out.mkdir(parents=True, exist_ok=True)

    subprocess.check_call([sys.executable, str(script), 'check', '--pack', str(local_pack)])
    # 1. One-epoch pilot sanity check
    subprocess.check_call([
        sys.executable, str(script), 'train', '--pack', str(local_pack), '--out', str(local_out),
        '--epochs', str(cfg.get('pilot_epochs', 1)), '--name', 'pilot',
        '--lr0', str(cfg.get('lr0', 0.001)), '--batch', str(cfg.get('batch', 16)), '--imgsz', str(cfg.get('imgsz', 1024))
    ])
    subprocess.check_call([sys.executable, str(script), 'predict', '--pack', str(local_pack), '--out', str(local_out), '--name', 'pilot'])
    # 2. Full fine-tuning
    run_name = cfg.get('run_name', 'own')
    subprocess.check_call([
        sys.executable, str(script), 'train', '--pack', str(local_pack), '--out', str(local_out),
        '--epochs', str(cfg.get('train_epochs', 40)), '--name', run_name,
        '--lr0', str(cfg.get('lr0', 0.001)), '--batch', str(cfg.get('batch', 16)), '--imgsz', str(cfg.get('imgsz', 1024))
    ])
    subprocess.check_call([sys.executable, str(script), 'predict', '--pack', str(local_pack), '--out', str(local_out), '--name', run_name])
    # Sync out/ and runs/<run_name>/weights/best.pt + results.csv to Google Drive
    shutil.copytree(local_out / 'out', res_dir / 'out', dirs_exist_ok=True)
    for rel in (f'runs/{run_name}/weights/best.pt', f'runs/{run_name}/results.csv', f'runs/{run_name}/args.yaml'):
        src = local_out / rel
        if src.is_file():
            dst = res_dir / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

for pname in SELECTED_PACKS:
    pdir = DRIVE_ROOT / 'packs' / pname
    rdir = DRIVE_ROOT / 'results' / pname
    rdir.mkdir(parents=True, exist_ok=True)
    meta = json.loads((pdir / 'colab-pack.json').read_text()) if (pdir / 'colab-pack.json').is_file() else {}
    ptype = meta.get('pack_type') or ('vllm-ocr' if (pdir / 'candidates.json').is_file() else 'generic')
    print(f'>>> Starting pack {pname} (type={ptype}) -> {rdir}', flush=True)
    if ptype == 'vllm-ocr':
        run_vllm_ocr_pack(pdir, rdir)
    elif ptype == 'yolo-marks':
        run_yolo_marks_pack(pdir, rdir)
    else:
        raise RuntimeError(f'Unsupported pack_type {ptype} in {pdir}')
    print(f'<<< Finished pack {pname}', flush=True)
print('ALL SELECTED PACKS FINISHED.')

In [ ]:
# 4. Completion summary
import json
from pathlib import Path

for pname in SELECTED_PACKS:
    rdir = DRIVE_ROOT / 'results' / pname
    print(f'=== {pname} ===')
    for status_file in sorted(rdir.rglob('*status.json')):
        rel = status_file.relative_to(rdir)
        st = json.loads(status_file.read_text())
        print(' ', rel, '->', {k: st.get(k) for k in ('state', 'done', 'total', 'errors', 'capped', 'eval_pages', 'total_boxes') if k in st})